In [1]:
import os
from pathlib import Path

import sqlglot
from sqlglot import exp, expressions

from src.utils.file_utils import parse_file_name
from src.migration.decomposer import SqlDecomposer, DecomposerWriter
from src.migration.metadata import MetadataProcessor
from src.migration.generator import PySparkGenerator
from src.paths import *

In [2]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

source_rules = load_all_source_rules()[source_name]

In [3]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = SqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = DecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = MetadataProcessor(source_rules, ai_fallback=False)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file, output_root)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_r_mhbos_m_client_crs\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 3a
   -> Khóa (Key) nhận diện được: client_no
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_r_mhbos_m_client_crs\metadata\com_r_mhbos_m_client_crs.yaml


In [4]:
step_file = Path(pipeline_config.get("pre_processing")[0]["file"])

step_file.read_text(encoding="utf-8")

"/* 1.0 backup month data without batch_date */\nCREATE TABLE IF NOT EXISTS ${com_schema}.r_mhbos_m_client_crs_bf\nSTORED AS PARQUET\nTBLPROPERTIES (\n  'parquet.compression'='SNAPPY',\n  'external.table.purge'='true'\n) AS\nSELECT\n  client_no,\n  controlling_name,\n  country_tax_residence,\n  tax_identification_no,\n  reason,\n  reason_remarks,\n  entity_type,\n  crs_tax_type,\n  etl_timestamp,\n  etl_dt,\n  part_id\nFROM ${com_schema}.r_mhbos_m_client_crs\nWHERE\n  etl_dt <> '${batch_date}' AND part_id = '${batch_yyyymm}';\n\n/* 3.1 truncate temp table */\nDROP TABLE ${com_schema}.r_mhbos_m_client_crs_bf;"

In [5]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = PySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Generated DDL at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_r_mhbos_m_client_crs\ddl\com_t_mhbos_m_client_crs.sql
Generated DML at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_r_mhbos_m_client_crs\dml\com_t_mhbos_m_client_crs.py
🎉 Hoàn tất toàn bộ Pipeline!


In [6]:
decomposed_script.temp_tables[0].ast_nodes[3].expression

IndexError: list index out of range

In [ ]:

# 3. Xây dựng vế TRÁI mới: TO_DATE(dl_record_updated_date)
new_left = exp.Anonymous(
    this="TO_DATE",
    expressions=[
        exp.Column(this=exp.Identifier(this="dl_record_updated_date", quoted=False))
    ]
)

# 4. Xây dựng vế PHẢI mới (Nested Anonymous functions)
# UNIX_TIMESTAMP('{batch_date}', 'yyyyMMdd')
unix_ts = exp.Anonymous(
    this="UNIX_TIMESTAMP",
    expressions=[
        exp.Literal.string("{batch_date}"),
        exp.Literal.string("yyyyMMdd")
    ]
)

# FROM_UNIXTIME(...)
from_unix = exp.Anonymous(this="FROM_UNIXTIME", expressions=[unix_ts])

# TO_DATE(...)
new_right = exp.Anonymous(this="TO_DATE", expressions=[from_unix])

# 5. Update giá trị bên trong EQ node
new_col = exp.Column(
    this=new_left,
    table=exp.Identifier(this=col_node.table) if col_node.table else None,
    expression=new_right
)


new_col

In [ ]:
node = sqlglot.parse("""
  SELECT
    mmc.*,
    ROW_NUMBER() OVER (PARTITION BY mmc.client_no ORDER BY mmc.part_id DESC) AS rn
  FROM com.r_mhbos_m_client AS mmc
  INNER JOIN filtered_clients AS fc
    ON mmc.client_no = fc.client_no
  WHERE
    mmc.part_id <= '${batch_date}'
;
""")

node

In [ ]:

node = decomposed_script.temp_tables[0].ast_nodes[3].copy()


for table_node in node.find_all(exp.Table):
    print(table_node)
    if table_node.name.startswith("r_"):
        new_name = table_node.name[2:]
        table_node.set("this", exp.Identifier(this=new_name, quoted=table_node.this.args.get("quoted", False)))
        table_node.set("db", exp.Identifier(this=exp.Parameter(this=exp.Var(this="batch_date"))))
        print("Renamed to: ", table_node)


In [ ]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))